**Installations of Libraries**



In [ ]:
!pip install tldextract
!pip install python-whois publicsuffix2 python-dateutil


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.4/107.4 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.0/117.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.0/89.0 kB 7.0 MB/s eta 0:00:00


***Base Dataset***

In [ ]:
import pandas as pd

# Load the dataset
df = pd.read_csv("phishing_site_urls.csv")

# Inspect structure
print(df.head())
print(df['Label'].value_counts())

# Map labels: bad -> 1 (phishing), good -> 0 (legit)
df['Label'] = df['Label'].map({'bad': 1, 'good': 0})

# Drop duplicates if any
df = df.drop_duplicates()

# Reset index
df = df.reset_index(drop=True)

print("Dataset shape after cleaning:", df.shape)
print(df.head())



                                                 URL Label
0  nobell.it/70ffb52d079109dca5664cce6f317373782/...   bad
1  www.dghjdgf.com/paypal.co.uk/cycgi-bin/webscrc...   bad
2  serviciosbys.com/paypal.cgi.bin.get-into.herf....   bad
3  mail.printakid.com/www.online.americanexpress....   bad
4  thewhiskeydregs.com/wp-content/themes/widescre...   bad
Label
good    392924
bad     156422
Name: count, dtype: int64
Dataset shape after cleaning: (507210, 2)
                                                 URL  Label
0  nobell.it/70ffb52d079109dca5664cce6f317373782/...    1.0
1  www.dghjdgf.com/paypal.co.uk/cycgi-bin/webscrc...    1.0
2  serviciosbys.com/paypal.cgi.bin.get-into.herf....    1.0
3  mail.printakid.com/www.online.americanexpress....    1.0
4  thewhiskeydregs.com/wp-content/themes/widescre...    1.0


**Features based on Complete URL**

In [ ]:
import re
import tldextract

# ===============================
# ---- URL Feature Extraction based on complete URL ----
# ===============================

# Regular expression to detect an email in URL
EMAIL_RE = re.compile(r"[A-Za-z0-9._%+\-]+@[A-Za-z0-9.\-]+\.[A-Za-z]{2,}")

def count_char(s, ch):
    return s.count(ch)

def qty_tilde(s):
    # Count both ASCII ~ and Unicode ˜
    return s.count("~") + s.count("˜")

def tld_len(url):
    ext = tldextract.extract(url if isinstance(url, str) else "")
    return len((ext.suffix or "").replace(".", ""))  # e.g., 'co.uk' → 4

def email_in_url(url):
    return int(bool(EMAIL_RE.search(url if isinstance(url, str) else "")))

def extract_url_features(url):
    s = str(url)
    return {
        "qty_dot_url": count_char(s, "."),
        "qty_hyphen_url": count_char(s, "-"),
        "qty_underline_url": count_char(s, "_"),
        "qty_slash_url": count_char(s, "/"),
        "qty_questionmark_url": count_char(s, "?"),
        "qty_equal_url": count_char(s, "="),
        "qty_at_url": count_char(s, "@"),
        "qty_and_url": count_char(s, "&"),
        "qty_exclamation_url": count_char(s, "!"),
        "qty_space_url": count_char(s, " "),
        "qty_tilde_url": qty_tilde(s),
        "qty_comma_url": count_char(s, ","),
        "qty_plus_url": count_char(s, "+"),
        "qty_asterisk_url": count_char(s, "*"),
        "qty_hashtag_url": count_char(s, "#"),
        "qty_dollar_url": count_char(s, "$"),
        "qty_percent_url": count_char(s, "%"),
        "qty_tld_url": tld_len(s),
        "length_url": len(s),
        "email_in_url": email_in_url(s),
    }

# Apply feature extraction to your dataset
url_features = df["URL"].apply(extract_url_features).apply(pd.Series)

# Combine with original dataframe
df_features = pd.concat([df, url_features], axis=1)

# Inspect
print("Features generated:", df_features.shape)
print(df_features.head())

# check column names
print(df_features.columns.tolist())


Features generated: (507210, 22)
                                                 URL  Label  qty_dot_url  \
0  nobell.it/70ffb52d079109dca5664cce6f317373782/...    1.0            6   
1  www.dghjdgf.com/paypal.co.uk/cycgi-bin/webscrc...    1.0            5   
2  serviciosbys.com/paypal.cgi.bin.get-into.herf....    1.0            7   
3  mail.printakid.com/www.online.americanexpress....    1.0            6   
4  thewhiskeydregs.com/wp-content/themes/widescre...    1.0            1   

   qty_hyphen_url  qty_underline_url  qty_slash_url  qty_questionmark_url  \
0               4                  4             10                     1   
1               2                  1              4                     0   
2               1                  0             11                     0   
3               0                  0              2                     0   
4               1                  0             10                     1   

   qty_equal_url  qty_at_url  qty_and_url  ... 

**Dataset attributes based on Domain URL**

In [ ]:
import re
import ipaddress
from urllib.parse import urlparse

# -------------------------
# Domain feature extraction
# -------------------------

VOWELS = set("aeiouAEIOU")

def _get_hostname(u: str) -> str:
    try:
        p = urlparse(str(u))
        # urlparse.hostname gives lowercase host, without userinfo/port
        return p.hostname or ""
    except Exception:
        return ""

def _is_ip(host: str) -> int:
    try:
        ipaddress.ip_address(host)
        return 1
    except Exception:
        return 0

def _count_char(s: str, ch: str) -> int:
    return s.count(ch)

def _qty_tilde(s: str) -> int:
    # Count both ASCII ~ and Unicode small tilde ˜
    return s.count("~") + s.count("˜")

def _qty_vowels(s: str) -> int:
    return sum(1 for c in s if c in VOWELS)

def extract_domain_features(url: str) -> dict:
    host = _get_hostname(url)
    s = host if isinstance(host, str) else ""
    s_low = s.lower()

    return {
        "qty_dot_domain": _count_char(s, "."),
        "qty_hyphen_domain": _count_char(s, "-"),
        "qty_underline_domain": _count_char(s, "_"),
        "qty_slash_domain": _count_char(s, "/"),
        "qty_questionmark_domain": _count_char(s, "?"),
        "qty_equal_domain": _count_char(s, "="),
        "qty_at_domain": _count_char(s, "@"),
        "qty_and_domain": _count_char(s, "&"),
        "qty_exclamation_domain": _count_char(s, "!"),
        "qty_space_domain": _count_char(s, " "),
        "qty_tilde_domain": _qty_tilde(s),
        "qty_comma_domain": _count_char(s, ","),
        "qty_plus_domain": _count_char(s, "+"),
        "qty_asterisk_domain": _count_char(s, "*"),
        "qty_hashtag_domain": _count_char(s, "#"),
        "qty_dollar_domain": _count_char(s, "$"),
        "qty_percent_domain": _count_char(s, "%"),
        "qty_vowels_domain": _qty_vowels(s),
        "domain_length": len(s),
        "domain_in_ip": _is_ip(s),
        # 1 if 'server' or 'client' appears anywhere in the domain
        "server_client_domain": int(("server" in s_low) or ("client" in s_low)),
    }

# Apply to your DataFrame (uses the URL column)
domain_feats = df["URL"].apply(extract_domain_features).apply(pd.Series)
df = pd.concat([df, domain_feats], axis=1)

print("Domain features added:", domain_feats.shape[1], "columns")
print(df.head())


✅ Domain features added: 21 columns
                                                 URL  Label  qty_dot_domain  \
0  nobell.it/70ffb52d079109dca5664cce6f317373782/...    1.0               0   
1  www.dghjdgf.com/paypal.co.uk/cycgi-bin/webscrc...    1.0               0   
2  serviciosbys.com/paypal.cgi.bin.get-into.herf....    1.0               0   
3  mail.printakid.com/www.online.americanexpress....    1.0               0   
4  thewhiskeydregs.com/wp-content/themes/widescre...    1.0               0   

   qty_hyphen_domain  qty_underline_domain  qty_slash_domain  \
0                  0                     0                 0   
1                  0                     0                 0   
2                  0                     0                 0   
3                  0                     0                 0   
4                  0                     0                 0   

   qty_questionmark_domain  qty_equal_domain  qty_at_domain  qty_and_domain  \
0                        

**Features based on URL directory**

In [ ]:
from urllib.parse import urlparse

def _get_directory(u: str) -> str:
    """Return directory/path from a URL (excluding domain, query, and fragment)."""
    try:
        p = urlparse(str(u))
        return p.path or ""
    except Exception:
        return ""

def extract_directory_features(url: str) -> dict:
    path = _get_directory(url)
    s = path if isinstance(path, str) else ""

    return {
        "qty_dot_directory": s.count("."),
        "qty_hyphen_directory": s.count("-"),
        "qty_underline_directory": s.count("_"),
        "qty_slash_directory": s.count("/"),
        "qty_questionmark_directory": s.count("?"),
        "qty_equal_directory": s.count("="),
        "qty_at_directory": s.count("@"),
        "qty_and_directory": s.count("&"),
        "qty_exclamation_directory": s.count("!"),
        "qty_space_directory": s.count(" "),
        "qty_tilde_directory": s.count("~") + s.count("˜"),
        "qty_comma_directory": s.count(","),
        "qty_plus_directory": s.count("+"),
        "qty_asterisk_directory": s.count("*"),
        "qty_hashtag_directory": s.count("#"),
        "qty_dollar_directory": s.count("$"),
        "qty_percent_directory": s.count("%"),
        "directory_length": len(s),
    }

# Apply to dataset
directory_feats = df["URL"].apply(extract_directory_features).apply(pd.Series)

# Append to main dataframe
df = pd.concat([df, directory_feats], axis=1)

print("Directory features added:", directory_feats.shape[1], "columns")
print(df.head())


✅ Directory features added: 18 columns
                                                 URL  Label  qty_dot_domain  \
0  nobell.it/70ffb52d079109dca5664cce6f317373782/...    1.0               0   
1  www.dghjdgf.com/paypal.co.uk/cycgi-bin/webscrc...    1.0               0   
2  serviciosbys.com/paypal.cgi.bin.get-into.herf....    1.0               0   
3  mail.printakid.com/www.online.americanexpress....    1.0               0   
4  thewhiskeydregs.com/wp-content/themes/widescre...    1.0               0   

   qty_hyphen_domain  qty_underline_domain  qty_slash_domain  \
0                  0                     0                 0   
1                  0                     0                 0   
2                  0                     0                 0   
3                  0                     0                 0   
4                  0                     0                 0   

   qty_questionmark_domain  qty_equal_domain  qty_at_domain  qty_and_domain  \
0                     

**URL based on file name**

In [ ]:
from urllib.parse import urlparse
import os

def _get_file_part(u: str) -> str:
    """
    Extract the file part from the URL path.
    Example:
        https://abc.com/dir/page.php?id=1  -> 'page.php'
        https://abc.com/folder/             -> ''
    """
    try:
        p = urlparse(str(u))
        path = p.path or ""
        # os.path.basename returns text after the last slash
        file_part = os.path.basename(path)
        return file_part
    except Exception:
        return ""

def extract_file_features(url: str) -> dict:
    f = _get_file_part(url)
    s = f if isinstance(f, str) else ""

    return {
        "qty_dot_file": s.count("."),
        "qty_hyphen_file": s.count("-"),
        "qty_underline_file": s.count("_"),
        "qty_slash_file": s.count("/"),
        "qty_questionmark_file": s.count("?"),
        "qty_equal_file": s.count("="),
        "qty_at_file": s.count("@"),
        "qty_and_file": s.count("&"),
        "qty_exclamation_file": s.count("!"),
        "qty_space_file": s.count(" "),
        "qty_tilde_file": s.count("~") + s.count("˜"),
        "qty_comma_file": s.count(","),
        "qty_plus_file": s.count("+"),
        "qty_asterisk_file": s.count("*"),
        "qty_hashtag_file": s.count("#"),
        "qty_dollar_file": s.count("$"),
        "qty_percent_file": s.count("%"),
        "file_length": len(s),
    }

# Apply to dataset
file_feats = df["URL"].apply(extract_file_features).apply(pd.Series)

# Merge with your main dataframe
df = pd.concat([df, file_feats], axis=1)

print("File features added:", file_feats.shape[1], "columns")
print(df.head())


File features added: 18 columns
                                                 URL  Label  qty_dot_domain  \
0  nobell.it/70ffb52d079109dca5664cce6f317373782/...    1.0               0   
1  www.dghjdgf.com/paypal.co.uk/cycgi-bin/webscrc...    1.0               0   
2  serviciosbys.com/paypal.cgi.bin.get-into.herf....    1.0               0   
3  mail.printakid.com/www.online.americanexpress....    1.0               0   
4  thewhiskeydregs.com/wp-content/themes/widescre...    1.0               0   

   qty_hyphen_domain  qty_underline_domain  qty_slash_domain  \
0                  0                     0                 0   
1                  0                     0                 0   
2                  0                     0                 0   
3                  0                     0                 0   
4                  0                     0                 0   

   qty_questionmark_domain  qty_equal_domain  qty_at_domain  qty_and_domain  \
0                        0   

**Features based on URL parameters**

In [ ]:
from urllib.parse import urlparse
import tldextract
import re

# Common TLD patterns to detect inside parameters
TLD_PATTERN = re.compile(r"\.(com|net|org|info|biz|edu|gov|io|co|uk|us|in|xyz|top|site|online|store|me|ai|dev|app)\b", re.IGNORECASE)

def _get_params(u: str) -> str:
    """Extract query string (part after '?')"""
    try:
        p = urlparse(str(u))
        return p.query or ""
    except Exception:
        return ""

def _tld_present_in_params(s: str) -> int:
    return int(bool(TLD_PATTERN.search(s)))

def _qty_params(s: str) -> int:
    # Count number of key=value pairs
    if not s:
        return 0
    parts = [p for p in s.split("&") if p.strip()]
    return len(parts)

def extract_params_features(url: str) -> dict:
    params = _get_params(url)
    s = params if isinstance(params, str) else ""

    return {
        "qty_dot_params": s.count("."),
        "qty_hyphen_params": s.count("-"),
        "qty_underline_params": s.count("_"),
        "qty_slash_params": s.count("/"),
        "qty_questionmark_params": s.count("?"),
        "qty_equal_params": s.count("="),
        "qty_at_params": s.count("@"),
        "qty_and_params": s.count("&"),
        "qty_exclamation_params": s.count("!"),
        "qty_space_param": s.count(" "),
        "qty_tilde_params": s.count("~") + s.count("˜"),
        "qty_comma_params": s.count(","),
        "qty_plus_params": s.count("+"),
        "qty_asterisk_params": s.count("*"),
        "qty_hashtag_params": s.count("#"),
        "qty_dollar_params": s.count("$"),
        "qty_percent_params": s.count("%"),
        "params_length": len(s),
        "tld_present_params": _tld_present_in_params(s),
        "qty_params": _qty_params(s),
    }

# Apply to dataset
params_feats = df["URL"].apply(extract_params_features).apply(pd.Series)

# Merge with main dataframe
df = pd.concat([df, params_feats], axis=1)

print(" Params features added:", params_feats.shape[1], "columns")
print(df.head())


 Params features added: 20 columns
                                                 URL  Label  qty_dot_domain  \
0  nobell.it/70ffb52d079109dca5664cce6f317373782/...    1.0               0   
1  www.dghjdgf.com/paypal.co.uk/cycgi-bin/webscrc...    1.0               0   
2  serviciosbys.com/paypal.cgi.bin.get-into.herf....    1.0               0   
3  mail.printakid.com/www.online.americanexpress....    1.0               0   
4  thewhiskeydregs.com/wp-content/themes/widescre...    1.0               0   

   qty_hyphen_domain  qty_underline_domain  qty_slash_domain  \
0                  0                     0                 0   
1                  0                     0                 0   
2                  0                     0                 0   
3                  0                     0                 0   
4                  0                     0                 0   

   qty_questionmark_domain  qty_equal_domain  qty_at_domain  qty_and_domain  \
0                        0

**Adding Benign Urls from Tranco**

In [ ]:
import pandas as pd

# Load Tranco file (no header)
tranco_df = pd.read_csv("tranco_7NZNX.csv", header=None, names=["Rank", "Domain"])

# Create full URLs by prefixing http:// (or https://)
tranco_df["URL"] = "http://" + tranco_df["Domain"]

# Assign label = 0 (benign)
tranco_df["Label"] = 0

# Keep only relevant columns
benign_df = tranco_df[["URL", "Label"]]

print(benign_df.head())
print("Total benign samples:", benign_df.shape[0])


                     URL  Label
0      http://google.com      0
1         http://mail.ru      0
2   http://microsoft.com      0
3    http://facebook.com      0
4  http://googleapis.com      0
Total benign samples: 1000000


**Attributes based on URL**

In [ ]:
import re
import pandas as pd
from urllib.parse import urlparse
# --- 20 URL-level lexical features ---
def extract_url_counts(url: str) -> dict:
    if not isinstance(url, str):
        return {}
    url = url.strip()

    signs = {
        "qty_dot_url": ".",
        "qty_hyphen_url": "-",
        "qty_underline_url": "_",
        "qty_slash_url": "/",
        "qty_questionmark_url": "?",
        "qty_equal_url": "=",
        "qty_at_url": "@",
        "qty_and_url": "&",
        "qty_exclamation_url": "!",
        "qty_space_url": " ",
        "qty_tilde_url": "~",
        "qty_comma_url": ",",
        "qty_plus_url": "+",
        "qty_asterisk_url": "*",
        "qty_hashtag_url": "#",
        "qty_dollar_url": "$",
        "qty_percent_url": "%",
    }

    feats = {k: url.count(ch) for k, ch in signs.items()}

    # TLD length
    try:
        host = (urlparse(url).hostname or "")
        feats["qty_tld_url"] = len(host.rsplit(".", 1)[-1]) if "." in host else 0
    except Exception:
        feats["qty_tld_url"] = 0

    # Total length
    feats["length_url"] = len(url)

    # Email-like pattern in URL
    feats["email_in_url"] = int(bool(re.search(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}", url)))

    return feats

# --- apply (no file writes) ---
url_lex_df = benign_df["URL"].apply(extract_url_counts).apply(pd.Series)



print("Feature columns:", list(url_lex_df.columns)[:5], "… (total:", url_lex_df.shape[1], ")")
print(url_lex_df.head())

Feature columns: ['qty_dot_url', 'qty_hyphen_url', 'qty_underline_url', 'qty_slash_url', 'qty_questionmark_url'] … (total: 20 )
   qty_dot_url  qty_hyphen_url  qty_underline_url  qty_slash_url  \
0            1               0                  0              2   
1            1               0                  0              2   
2            1               0                  0              2   
3            1               0                  0              2   
4            1               0                  0              2   

   qty_questionmark_url  qty_equal_url  qty_at_url  qty_and_url  \
0                     0              0           0            0   
1                     0              0           0            0   
2                     0              0           0            0   
3                     0              0           0            0   
4                     0              0           0            0   

   qty_exclamation_url  qty_space_url  qty_tilde_url  qty_co

**Dataset attributes based on domain URL**

In [ ]:
import pandas as pd
import re
from urllib.parse import urlparse
import ipaddress

# ---------- Function: extract_domain_features ----------
def extract_domain_features(url: str) -> dict:
    """Extract 21 domain-level lexical features"""
    if not isinstance(url, str):
        return {}

    url = url.strip()
    features = {}

    # Parse domain (hostname)
    try:
        domain = urlparse(url).hostname or ""
    except Exception:
        domain = ""

    # Symbols to count
    signs = {
        "qty_dot_domain": ".",
        "qty_hyphen_domain": "-",
        "qty_underline_domain": "_",
        "qty_slash_domain": "/",
        "qty_questionmark_domain": "?",
        "qty_equal_domain": "=",
        "qty_at_domain": "@",
        "qty_and_domain": "&",
        "qty_exclamation_domain": "!",
        "qty_space_domain": " ",
        "qty_tilde_domain": "~",
        "qty_comma_domain": ",",
        "qty_plus_domain": "+",
        "qty_asterisk_domain": "*",
        "qty_hashtag_domain": "#",
        "qty_dollar_domain": "$",
        "qty_percent_domain": "%",
    }

    # count each special character
    for feat, ch in signs.items():
        features[feat] = domain.count(ch)

    # number of vowels
    features["qty_vowels_domain"] = sum(domain.lower().count(v) for v in "aeiou")

    # domain length
    features["domain_length"] = len(domain)

    # check if domain is an IP address (IPv4 or IPv6)
    try:
        ipaddress.ip_address(domain)
        features["domain_in_ip"] = 1
    except Exception:
        features["domain_in_ip"] = 0

    # "server" or "client" present in domain
    lower_domain = domain.lower()
    features["server_client_domain"] = int(("server" in lower_domain) or ("client" in lower_domain))

    return features

# ---------- Example Usage ----------
domain_features_df = benign_df["URL"].apply(extract_domain_features).apply(pd.Series)

print(" Extracted domain-level features:", domain_features_df.shape[1])
print(domain_features_df.head())


 Extracted domain-level features: 21
   qty_dot_domain  qty_hyphen_domain  qty_underline_domain  qty_slash_domain  \
0               1                  0                     0                 0   
1               1                  0                     0                 0   
2               1                  0                     0                 0   
3               1                  0                     0                 0   
4               1                  0                     0                 0   

   qty_questionmark_domain  qty_equal_domain  qty_at_domain  qty_and_domain  \
0                        0                 0              0               0   
1                        0                 0              0               0   
2                        0                 0              0               0   
3                        0                 0              0               0   
4                        0                 0              0               0   

   qty_

**Dataset attributes based on URL directory**

In [ ]:
import pandas as pd
from urllib.parse import urlparse

# ---------- Function: extract_directory_features ----------
def extract_directory_features(url: str) -> dict:
    """Extract 18 directory-level lexical features"""
    if not isinstance(url, str):
        return {}

    url = url.strip()
    features = {}

    # Extract directory (path) part from URL
    try:
        path = urlparse(url).path or ""
    except Exception:
        path = ""

    # Signs to count
    signs = {
        "qty_dot_directory": ".",
        "qty_hyphen_directory": "-",
        "qty_underline_directory": "_",
        "qty_slash_directory": "/",
        "qty_questionmark_directory": "?",
        "qty_equal_directory": "=",
        "qty_at_directory": "@",
        "qty_and_directory": "&",
        "qty_exclamation_directory": "!",
        "qty_space_directory": " ",
        "qty_tilde_directory": "~",
        "qty_comma_directory": ",",
        "qty_plus_directory": "+",
        "qty_asterisk_directory": "*",
        "qty_hashtag_directory": "#",
        "qty_dollar_directory": "$",
        "qty_percent_directory": "%",
    }

    # Count occurrences for each symbol
    for feat, ch in signs.items():
        features[feat] = path.count(ch)

    # Length of directory
    features["directory_length"] = len(path)

    return features

# ---------- Example Usage ----------
directory_features_df = benign_df["URL"].apply(extract_directory_features).apply(pd.Series)

print(" Extracted directory-level features:", directory_features_df.shape[1])
print(directory_features_df.head())


 Extracted directory-level features: 18
   qty_dot_directory  qty_hyphen_directory  qty_underline_directory  \
0                  0                     0                        0   
1                  0                     0                        0   
2                  0                     0                        0   
3                  0                     0                        0   
4                  0                     0                        0   

   qty_slash_directory  qty_questionmark_directory  qty_equal_directory  \
0                    0                           0                    0   
1                    0                           0                    0   
2                    0                           0                    0   
3                    0                           0                    0   
4                    0                           0                    0   

   qty_at_directory  qty_and_directory  qty_exclamation_directory  \
0            

**Dataset attributes based on URL file name.**

In [ ]:
import pandas as pd
from urllib.parse import urlparse

# ---------- Function: extract_file_features ----------
def extract_file_features(url: str) -> dict:
    """Extract 18 lexical features from the file part of a URL"""
    if not isinstance(url, str):
        return {}

    url = url.strip()
    features = {}

    # Extract file part (last segment of path)
    try:
        path = urlparse(url).path or ""
        file_part = path.split("/")[-1] if path else ""
    except Exception:
        file_part = ""

    # Characters to count
    signs = {
        "qty_dot_file": ".",
        "qty_hyphen_file": "-",
        "qty_underline_file": "_",
        "qty_slash_file": "/",
        "qty_questionmark_file": "?",
        "qty_equal_file": "=",
        "qty_at_file": "@",
        "qty_and_file": "&",
        "qty_exclamation_file": "!",
        "qty_space_file": " ",
        "qty_tilde_file": "~",
        "qty_comma_file": ",",
        "qty_plus_file": "+",
        "qty_asterisk_file": "*",
        "qty_hashtag_file": "#",
        "qty_dollar_file": "$",
        "qty_percent_file": "%",
    }

    # Count each character’s occurrences in the file name
    for feat, ch in signs.items():
        features[feat] = file_part.count(ch)

    # Length of file part
    features["file_length"] = len(file_part)

    return features

# ---------- Example Usage ----------

file_features_df = benign_df["URL"].apply(extract_file_features).apply(pd.Series)

print("Extracted file-level features:", file_features_df.shape[1])
print(file_features_df.head())


Extracted file-level features: 18
   qty_dot_file  qty_hyphen_file  qty_underline_file  qty_slash_file  \
0             0                0                   0               0   
1             0                0                   0               0   
2             0                0                   0               0   
3             0                0                   0               0   
4             0                0                   0               0   

   qty_questionmark_file  qty_equal_file  qty_at_file  qty_and_file  \
0                      0               0            0             0   
1                      0               0            0             0   
2                      0               0            0             0   
3                      0               0            0             0   
4                      0               0            0             0   

   qty_exclamation_file  qty_space_file  qty_tilde_file  qty_comma_file  \
0                     0        

**Dataset attributes based on URL parameters**

In [ ]:
import re
from urllib.parse import urlparse

# ---------- Params-level features (20) ----------
def extract_params_features(url: str) -> dict:
    """
    Features computed on the query string (after '?'):
      - qty_*_params for the listed symbols
      - params_length (len of query)
      - tld_present_params (1 if a domain/TLD-like token appears in params)
      - qty_params (number of key-value pairs)
    """
    if not isinstance(url, str):
        return {}

    try:
        query = urlparse(url.strip()).query or ""
    except Exception:
        query = ""

    feats = {}

    # character counts
    signs = {
        "qty_dot_params": ".",
        "qty_hyphen_params": "-",
        "qty_underline_params": "_",
        "qty_slash_params": "/",
        "qty_questionmark_params": "?",
        "qty_equal_params": "=",
        "qty_at_params": "@",
        "qty_and_params": "&",
        "qty_exclamation_params": "!",
        "qty_space_params": " ",
        "qty_tilde_params": "~",
        "qty_comma_params": ",",
        "qty_plus_params": "+",
        "qty_asterisk_params": "*",
        "qty_hashtag_params": "#",
        "qty_dollar_params": "$",
        "qty_percent_params": "%",
    }
    for col, ch in signs.items():
        feats[col] = query.count(ch)

    # total length of parameters string
    feats["params_length"] = len(query)

    # tld_present_params: check for domain-like tokens in params (e.g., example.com, a.b.co.uk)
    domain_like_pat = re.compile(r"\b(?:[a-z0-9-]+\.)+(?:[a-z]{2,24})\b", re.IGNORECASE)
    feats["tld_present_params"] = int(bool(domain_like_pat.search(query)))

    # qty_params: number of parameters (split on '&', ignore empties)
    if query:
        parts = [p for p in query.split("&") if p]  # keep empty-safe
        feats["qty_params"] = len(parts)
    else:
        feats["qty_params"] = 0

    return feats

# --------- ----------
params_df = benign_df["URL"].apply(extract_params_features).apply(pd.Series)
print("Params features:", params_df.shape[1], "columns")
print(params_df.head())


Params features: 20 columns
   qty_dot_params  qty_hyphen_params  qty_underline_params  qty_slash_params  \
0               0                  0                     0                 0   
1               0                  0                     0                 0   
2               0                  0                     0                 0   
3               0                  0                     0                 0   
4               0                  0                     0                 0   

   qty_questionmark_params  qty_equal_params  qty_at_params  qty_and_params  \
0                        0                 0              0               0   
1                        0                 0              0               0   
2                        0                 0              0               0   
3                        0                 0              0               0   
4                        0                 0              0               0   

   qty_exclamati

**Building dataset based on Algorithm 1**
Algorithm 1 Feature extraction process

Input: URLs  Array of URLs.

Input: signs  Array of signs to count.

Output: dataset.csv  Output csv document.

1: i ← 0

2: totalURLs ← lenght(URLs)  Get the number of URLs in array.

3: while i < totalURLs do

4: url ← URLs(i)

5: countsUrl ← getCounts(url, signs)  Get signs and character counts.

6: for substring in splitURL(url) do  Iterate through the sub-strings of URL.

7: countsSubstring ← getCounts(substrings, signs)  Get signs and character counts.

8: end for

9: measuredFeatures ← f etchFeatures(url)  Get features from external services.

10: toCsv(countsUrl, countsSubstring, measuredFeatures)  Append row to csv.

11: i ← i + 1

12: end while

In [ ]:
# === FINAL: Algorithm 1 Full Dataset Feature Extraction ===
import time, concurrent.futures, requests, pandas as pd, re
from urllib.parse import urlparse, urlunparse, quote
from requests.utils import requote_uri

# -----------------------------
# Config
# -----------------------------
CONNECTIONS = 100             # parallel workers
HTTP_CONNECT_TIMEOUT = 2.0    # seconds to connect
HTTP_READ_TIMEOUT = 2.0       # seconds to read
TIMEOUT = (HTTP_CONNECT_TIMEOUT, HTTP_READ_TIMEOUT)
UA = {"User-Agent": "Mozilla/5.0 (compatible; MScFeatureExtractor/1.0)"}
OUTPUT_FILE = "dataset_full.csv"

SIGNS = [".","-","_","/","?","=","@","&","!"," ","~","˜",",","+","*","#","$","%"]
SIGN_LABELS = {
    ".":"dot","-":"hyphen","_":"underline","/":"slash","?":"questionmark","=":"equal",
    "@":"at","&":"and","!":"exclamation"," ":"space","~":"tilde","˜":"tilde",",":"comma",
    "+":"plus","*":"asterisk","#":"hashtag","$":"dollar","%":"percent"
}

# -----------------------------
# Helpers
# -----------------------------
def normalize_url(u: str) -> str:
    s = (str(u) or "").strip()
    if not s: return ""
    if s.startswith("//"): s = "http:" + s
    if "://" not in s: s = "http://" + s
    p = urlparse(s)
    if not p.netloc and p.path:
        p = urlparse(f"{p.scheme}://{p.path}")
    host = p.hostname or ""
    port = p.port
    user = p.username or ""
    pwd  = p.password or ""
    userinfo = ""
    if user:
        userinfo = quote(user, safe="")
        if pwd: userinfo += ":" + quote(pwd, safe="")
        userinfo += "@"
    if ":" in host and not (host.startswith("[") and host.endswith("]")):
        host = f"[{host}]"
    netloc = userinfo + host + (f":{port}" if port else "")
    return requote_uri(urlunparse((p.scheme, netloc, p.path or "", p.params or "", p.query or "", p.fragment or "")))

def splitURL(url: str) -> dict:
    try:
        p = urlparse(normalize_url(url))
        directory = p.path or ""
        return {
            "domain":   p.hostname or "",
            "directory": directory,
            "file":      directory.split("/")[-1] if directory else "",
            "params":    p.query or "",
            "fragment":  p.fragment or "",
        }
    except Exception:
        return {"domain":"","directory":"","file":"","params":"","fragment":""}

def getCounts(text: str, signs: list, scope: str) -> dict:
    s = text if isinstance(text, str) else ""
    out = {}
    tilde_total, done = s.count("~") + s.count("˜"), False
    for ch in signs:
        token = SIGN_LABELS.get(ch, f"u{ord(ch)}"); col = f"qty_{token}_{scope}"
        if ch in ("~","˜"):
            if not done:
                out[col] = tilde_total; done = True
        else:
            out[col] = s.count(ch)
    if scope == "url": out["length_url"] = len(s)
    else: out[f"{scope}_length"] = len(s)
    return out

# -----------------------------
# Parallel HTTP HEAD (Jack’s fast approach)
# -----------------------------
def _head_worker(url: str) -> dict:
    res = {
        "time_response": float("nan"),
        "qty_redirects": 0,
        "http_status": float("nan"),
        "server_header": "",
        "tls_ssl_certificate": 1 if normalize_url(url).lower().startswith("https") else 0,
    }
    try:
        t0 = time.perf_counter()
        r = requests.head(normalize_url(url), headers=UA, timeout=TIMEOUT, allow_redirects=True)
        res["time_response"] = int((time.perf_counter() - t0) * 1000)
        res["qty_redirects"] = len(r.history)
        res["http_status"]   = r.status_code
        res["server_header"] = r.headers.get("Server","")
    except requests.exceptions.Timeout:
        pass
    except Exception:
        pass
    return res

def fetch_features_parallel(urls, max_workers=CONNECTIONS):
    out = [None] * len(urls)
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = {ex.submit(_head_worker, u): idx for idx, u in enumerate(urls)}
        done = 0; total = len(urls); step = max(1, total // 10)
        for fut in concurrent.futures.as_completed(futures):
            idx = futures[fut]
            try:
                out[idx] = fut.result()
            except Exception:
                out[idx] = _head_worker("")
            done += 1
            if done % step == 0 or done == total:
                print(f"  HEAD progress: {done}/{total}")
    return out

# -----------------------------
# Algorithm 1 Runner (full dataset)
# -----------------------------
def run_full_algorithm(input_csv="phishing_site_urls.csv", url_col=None, output_csv=OUTPUT_FILE):
    df = pd.read_csv(input_csv)

    # detect URL column
    if url_col is None:
        for c in ["URL","url","Url","Domain","domain"]:
            if c in df.columns:
                url_col = c; break
        if url_col is None:
            raise ValueError("No URL column found.")

    # Label mapping
    if "Label" in df.columns:
        df["Label"] = df["Label"].map({"bad":1,"good":0}).fillna(df["Label"])

    urls = df[url_col].astype(str).tolist()
    total = len(urls)
    print(f"\n Running Algorithm 1 on full dataset ({total} URLs)...")

    # Step 9 (external): parallel HEADs
    head_feats = fetch_features_parallel(urls, max_workers=CONNECTIONS)

    # Steps 5–8 (URL + substrings)
    rows = []
    i = 0
    while i < total:
        url = urls[i]
        row = {"url": url}
        if "Label" in df.columns:
            row["Label"] = df.loc[i, "Label"]

        # full URL counts
        row.update(getCounts(url, SIGNS, "url"))
        # split and count parts
        for scope, sub in splitURL(url).items():
            row.update(getCounts(sub, SIGNS, scope))
        # cached HEAD results
        row.update(head_feats[i] if head_feats[i] else _head_worker(""))

        rows.append(row)
        if i % max(1, total // 10) == 0 or i == total - 1:
            print(f"  Progress: {i+1}/{total}")
        i += 1

    out_df = pd.DataFrame(rows)
    out_df.to_csv(output_csv, index=False)
    print(f"\n Extracted {total} URLs → {output_csv}")
    display(out_df.head(100))
    return out_df

# -----------------------------
# Run for entire dataset
# -----------------------------
df_full = run_full_algorithm("phishing_site_urls.csv", output_csv="dataset_full.csv")


/tmp/ipython-input-1296916982.py:133: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["Label"] = df["Label"].map({"bad":1,"good":0}).fillna(df["Label"])



🚀 Running Algorithm 1 on full dataset (549360 URLs)...
  HEAD progress: 54936/549360


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/urllib3/connection.py", line 568, in getresponse
    assert_header_parsing(httplib_response.msg)
  File "/usr/local/lib/python3.12/dist-packages/urllib3/util/response.py", line 88, in assert_header_parsing
    raise HeaderParsingError(defects=defects, unparsed_data=unparsed_data)
urllib3.exceptions.HeaderParsingError: [MissingHeaderBodySeparatorDefect()], unparsed data: 'last modified header response: If-Modified-Since\r\nX-XSS-Protection: mode=block\r\nDate: Wed, 15 Oct 2025 10:26:49 GMT\r\n\r\n'
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/urllib3/connection.py", line 568, in getresponse
    assert_header_parsing(httplib_response.msg)
  File "/usr/local/lib/python3.12/dist-packages/urllib3/util/response.py", line 88, in assert_header_parsing
    raise HeaderParsingError(defects=defects, unparsed_data=unparsed_data)
urllib3.exceptions.HeaderParsingError: [MissingHea

  HEAD progress: 109872/549360
  HEAD progress: 164808/549360
  HEAD progress: 219744/549360
  HEAD progress: 274680/549360
  HEAD progress: 329616/549360


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/urllib3/connection.py", line 568, in getresponse
    assert_header_parsing(httplib_response.msg)
  File "/usr/local/lib/python3.12/dist-packages/urllib3/util/response.py", line 88, in assert_header_parsing
    raise HeaderParsingError(defects=defects, unparsed_data=unparsed_data)
urllib3.exceptions.HeaderParsingError: [MissingHeaderBodySeparatorDefect()], unparsed data: 'X-Content-Type-Optionx-frame-options : SAMEORIGIN"\r\nx-xss-protection: 0\r\n\r\n'


...

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/urllib3/connection.py", line 568, in getresponse
    assert_header_parsing(httplib_response.msg)
  File "/usr/local/lib/python3.12/dist-packages/urllib3/util/response.py", line 88, in assert_header_parsing
    raise HeaderParsingError(defects=defects, unparsed_data=unparsed_data)
urllib3.exceptions.HeaderParsingError: [MissingHeaderBodySeparatorDefect()], unparsed data: 'powered by: STFU.net\r\nX-Powered-By: ASP.NET\r\nDate: Wed, 15 Oct 2025 11:31:00 GMT\r\n\r\n'


  HEAD progress: 439488/549360


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/urllib3/connection.py", line 568, in getresponse
    assert_header_parsing(httplib_response.msg)
  File "/usr/local/lib/python3.12/dist-packages/urllib3/util/response.py", line 88, in assert_header_parsing
    raise HeaderParsingError(defects=defects, unparsed_data=unparsed_data)
urllib3.exceptions.HeaderParsingError: [MissingHeaderBodySeparatorDefect()], unparsed data: 'X-Content-Type-Optionx-frame-options : SAMEORIGIN"\r\nx-xss-protection: 0\r\n\r\n'
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/urllib3/connection.py", line 568, in getresponse
    assert_header_parsing(httplib_response.msg)
  File "/usr/local/lib/python3.12/dist-packages/urllib3/util/response.py", line 88, in assert_header_parsing
    raise HeaderParsingError(defects=defects, unparsed_data=unparsed_data)
urllib3.exceptions.HeaderParsingError: [MissingHeaderBodySeparatorDefect()], unparsed data: 'X-C

  HEAD progress: 494424/549360
  HEAD progress: 549360/549360
  Progress: 1/549360
  Progress: 54937/549360
  Progress: 109873/549360
  Progress: 164809/549360
  Progress: 219745/549360
  Progress: 274681/549360
  Progress: 329617/549360
  Progress: 384553/549360
  Progress: 439489/549360
  Progress: 494425/549360
  Progress: 549360/549360

 Extracted 549360 URLs → dataset_full.csv


,url,Label,qty_dot_url,qty_hyphen_url,qty_underline_url,qty_slash_url,qty_questionmark_url,qty_equal_url,qty_at_url,qty_and_url,...,qty_asterisk_fragment,qty_hashtag_fragment,qty_dollar_fragment,qty_percent_fragment,fragment_length,time_response,qty_redirects,http_status,server_header,tls_ssl_certificate
0,nobell.it/70ffb52d079109dca5664cce6f317373782/...,1.0,6,4,4,10,1,4,0,3,...,0,0,0,0,0,NaN,0,NaN,,0
1,www.dghjdgf.com/paypal.co.uk/cycgi-bin/webscrc...,1.0,5,2,1,4,0,2,0,1,...,0,0,0,0,0,NaN,0,NaN,,0
2,serviciosbys.com/paypal.cgi.bin.get-into.herf....,1.0,7,1,0,11,0,0,0,0,...,0,0,0,0,0,NaN,0,NaN,,0
3,mail.printakid.com/www.online.americanexpress....,1.0,6,0,0,2,0,0,0,0,...,0,0,0,0,0,NaN,0,NaN,,0
4,thewhiskeydregs.com/wp-content/themes/widescre...,1.0,1,1,0,10,1,0,0,0,...,0,0,0,0,0,NaN,0,NaN,,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,rsaxena.5gbfree.com/facebook.html,1.0,3,0,0,1,0,0,0,0,...,0,0,0,0,0,NaN,0,NaN,,0
96,steamcommunity-giveaway.my3gb.com,1.0,2,1,0,0,0,0,0,0,...,0,0,0,0,0,158.0,0,200.0,,0
97,steamglfts.h16.ru,1.0,2,0,0,0,0,0,0,0,...,0,0,0,0,0,NaN,0,NaN,,0
98,steamglfts.hut2.ru/,1.0,2,0,0,1,0,0,0,0,...,0,0,0,0,0,1236.0,1,200.0,nginx/1.24.0 (Ubuntu),0


In [ ]:
from google.colab import files
files.download("dataset_full.csv")
#------Dataset for Phishing URL-----#

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# === Algorithm 1 for TRANCO dataset (parallel HTTP + CSV output) ===
import time, concurrent.futures, requests, pandas as pd
from urllib.parse import urlparse, urlunparse, quote
from requests.utils import requote_uri

# ------------- config -------------
TRONCO_FILE = "tranco_7NZNX.csv"   # <-- change if needed
OUTPUT_10   = "tranco_dataset_10.csv"
OUTPUT_100  = "tranco_dataset_100.csv"
OUTPUT_FULL = "tranco_dataset_full.csv"

CONNECTIONS = 100             # parallel workers
HTTP_CONNECT_TIMEOUT = 2.0    # seconds to connect
HTTP_READ_TIMEOUT    = 2.0    # seconds to read
TIMEOUT = (HTTP_CONNECT_TIMEOUT, HTTP_READ_TIMEOUT)
UA = {"User-Agent": "Mozilla/5.0 (compatible; MScTrancoExtractor/1.0)"}

SIGNS = [".","-","_","/","?","=","@","&","!"," ","~","˜",",","+","*","#","$","%"]
SIGN_LABELS = {
    ".":"dot","-":"hyphen","_":"underline","/":"slash","?":"questionmark","=":"equal",
    "@":"at","&":"and","!":"exclamation"," ":"space","~":"tilde","˜":"tilde",",":"comma",
    "+":"plus","*":"asterisk","#":"hashtag","$":"dollar","%":"percent"
}

# ------------- helpers -------------
def normalize_url(u: str) -> str:
    s = (str(u) or "").strip()
    if not s: return ""
    if s.startswith("//"): s = "http:" + s
    if "://" not in s: s = "http://" + s
    p = urlparse(s)
    if not p.netloc and p.path:
        p = urlparse(f"{p.scheme}://{p.path}")
    host = p.hostname or ""
    port = p.port
    user = p.username or ""
    pwd  = p.password or ""
    userinfo = ""
    if user:
        userinfo = quote(user, safe="")
        if pwd: userinfo += ":" + quote(pwd, safe="")
        userinfo += "@"
    if ":" in host and not (host.startswith("[") and host.endswith("]")):
        host = f"[{host}]"
    netloc = userinfo + host + (f":{port}" if port else "")
    return requote_uri(urlunparse((p.scheme, netloc, p.path or "", p.params or "", p.query or "", p.fragment or "")))

def splitURL(url: str) -> dict:
    try:
        p = urlparse(normalize_url(url))
        directory = p.path or ""
        return {
            "domain":   p.hostname or "",
            "directory": directory,
            "file":      directory.split("/")[-1] if directory else "",
            "params":    p.query or "",
            "fragment":  p.fragment or "",
        }
    except Exception:
        return {"domain":"","directory":"","file":"","params":"","fragment":""}

def getCounts(text: str, signs: list, scope: str) -> dict:
    s = text if isinstance(text, str) else ""
    out = {}
    tilde_total, done = s.count("~") + s.count("˜"), False
    for ch in signs:
        token = SIGN_LABELS.get(ch, f"u{ord(ch)}"); col = f"qty_{token}_{scope}"
        if ch in ("~","˜"):
            if not done:
                out[col] = tilde_total; done = True
        else:
            out[col] = s.count(ch)
    if scope == "url": out["length_url"] = len(s)
    else: out[f"{scope}_length"] = len(s)
    return out

# ------------- parallel HTTP HEAD -------------
def _head_worker(url: str) -> dict:
    res = {
        "time_response": float("nan"),
        "qty_redirects": 0,
        "http_status": float("nan"),
        "server_header": "",
        "tls_ssl_certificate": 1 if normalize_url(url).lower().startswith("https") else 0,
    }
    try:
        t0 = time.perf_counter()
        r = requests.head(normalize_url(url), headers=UA, timeout=TIMEOUT, allow_redirects=True)
        res["time_response"] = int((time.perf_counter() - t0) * 1000)
        res["qty_redirects"] = len(r.history)
        res["http_status"]   = r.status_code
        res["server_header"] = r.headers.get("Server","")
    except requests.exceptions.Timeout:
        pass
    except Exception:
        pass
    return res

def fetch_features_parallel(urls, max_workers=CONNECTIONS):
    out = [None] * len(urls)
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = {ex.submit(_head_worker, u): idx for idx, u in enumerate(urls)}
        done = 0; total = len(urls); step = max(1, total // 10)
        for fut in concurrent.futures.as_completed(futures):
            idx = futures[fut]
            try:
                out[idx] = fut.result()
            except Exception:
                out[idx] = _head_worker("")
            done += 1
            if done % step == 0 or done == total:
                print(f"  HEAD progress: {done}/{total}")
    return out

# ------------- Algorithm 1 runner for Tranco -------------
def run_tranco_algorithm(tranco_csv=TRONCO_FILE, limit=None, output_csv="tranco_dataset.csv"):
    # Load tranco (Rank,Domain) -> URL + Label=0
    tdf = pd.read_csv(tranco_csv, header=None, names=["Rank","Domain"])
    tdf["URL"] = "http://" + tdf["Domain"].astype(str)
    tdf["Label"] = 0  # benign
    urls = tdf["URL"].astype(str).tolist()
    if limit is not None: urls = urls[:limit]
    total = len(urls)
    print(f"\n Tranco Algorithm 1 on {total} URLs (parallel HEAD={CONNECTIONS}, timeout={TIMEOUT})")

    # Step 9 (external) in parallel first
    head_feats = fetch_features_parallel(urls, max_workers=CONNECTIONS)

    # Steps 5–8 + TLD extraction
    rows = []
    i = 0
    while i < total:
        url = urls[i]
        row = {"url": url, "Label": 0}

        # URL-wide counts
        row.update(getCounts(url, SIGNS, "url"))

        # substrings (domain, directory, file, params, fragment)
        parts = splitURL(url)
        for scope, sub in parts.items():
            row.update(getCounts(sub, SIGNS, scope))

        # explicit TLD + its length
        dom = parts["domain"]
        row["tld"] = dom.split(".")[-1] if "." in dom else ""
        row["qty_tld_url"] = len(row["tld"])

        # cached HEAD metrics
        row.update(head_feats[i] if head_feats[i] else _head_worker(""))

        rows.append(row)
        if i % max(1, total // 10) == 0 or i == total - 1:
            print(f"  Build progress: {i+1}/{total}")
        i += 1

    out_df = pd.DataFrame(rows)
    out_df.to_csv(output_csv, index=False)
    print(f"\nDone! Wrote {total} rows → {output_csv}")
    display(out_df.head(10))
    return out_df

# ---------------- RUNS ----------------
# First 10
#df_tranco_10  = run_tranco_algorithm(TRONCO_FILE, limit=10,  output_csv=OUTPUT_10)

# First 100
#df_tranco_100 = run_tranco_algorithm(TRONCO_FILE, limit=100, output_csv=OUTPUT_100)

# Full dataset
df_tranco_full = run_tranco_algorithm(TRONCO_FILE, limit=None, output_csv=OUTPUT_FULL)



🚀 Tranco Algorithm 1 on 1000000 URLs (parallel HEAD=100, timeout=(2.0, 2.0))


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/urllib3/connection.py", line 568, in getresponse
    assert_header_parsing(httplib_response.msg)
  File "/usr/local/lib/python3.12/dist-packages/urllib3/util/response.py", line 88, in assert_header_parsing
    raise HeaderParsingError(defects=defects, unparsed_data=unparsed_data)
urllib3.exceptions.HeaderParsingError: [MissingHeaderBodySeparatorDefect()], unparsed data: "X-Content-Type-Options : nosniff\r\nContent-Security-Policy: default-src https:; script-src blob: 'unsafe-inline' 'unsafe-eval' 'self' https://*.uservoice.com https://maps.googleapis.com https://faronics.kayako.com/ https://apis.google.com/; connect-src https: 'self' ws:; img-src blob: https: 'self' data:; style-src 'unsafe-inline' 'self' https://fonts.googleapis.com; font-src https:;\r\nDate: Wed, 15 Oct 2025 12:26:35 GMT\r\n\r\n"
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/urllib3/connection.py", 

  HEAD progress: 100000/1000000


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/urllib3/connection.py", line 568, in getresponse
    assert_header_parsing(httplib_response.msg)
  File "/usr/local/lib/python3.12/dist-packages/urllib3/util/response.py", line 88, in assert_header_parsing
    raise HeaderParsingError(defects=defects, unparsed_data=unparsed_data)
urllib3.exceptions.HeaderParsingError: [MissingHeaderBodySeparatorDefect()], unparsed data: '404 Not Found: \r\nContent-Encoding: gzip\r\n\r\n'
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/urllib3/connection.py", line 568, in getresponse
    assert_header_parsing(httplib_response.msg)
  File "/usr/local/lib/python3.12/dist-packages/urllib3/util/response.py", line 88, in assert_header_parsing
    raise HeaderParsingError(defects=defects, unparsed_data=unparsed_data)
urllib3.exceptions.HeaderParsingError: [MissingHeaderBodySeparatorDefect()], unparsed data: 'X-Content-Type-Optionx-frame-option

  HEAD progress: 200000/1000000


...

  HEAD progress: 800000/1000000


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/urllib3/connection.py", line 568, in getresponse
    assert_header_parsing(httplib_response.msg)
  File "/usr/local/lib/python3.12/dist-packages/urllib3/util/response.py", line 88, in assert_header_parsing
    raise HeaderParsingError(defects=defects, unparsed_data=unparsed_data)
urllib3.exceptions.HeaderParsingError: [MissingHeaderBodySeparatorDefect()], unparsed data: 'Access-Control-Allow-Origin : *\r\nAccess-Control-Allow-Origin-Methods : X-Requested-With, Content-Type\r\nKeep-Alive: timeout=5, max=100\r\nConnection: Keep-Alive\r\nContent-Type: text/html;charset=utf-8\r\n\r\n'
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/urllib3/connection.py", line 568, in getresponse
    assert_header_parsing(httplib_response.msg)
  File "/usr/local/lib/python3.12/dist-packages/urllib3/util/response.py", line 88, in assert_header_parsing
    raise HeaderParsingError(defects=def

  HEAD progress: 900000/1000000


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/urllib3/connection.py", line 568, in getresponse
    assert_header_parsing(httplib_response.msg)
  File "/usr/local/lib/python3.12/dist-packages/urllib3/util/response.py", line 88, in assert_header_parsing
    raise HeaderParsingError(defects=defects, unparsed_data=unparsed_data)
urllib3.exceptions.HeaderParsingError: [MissingHeaderBodySeparatorDefect()], unparsed data: 'Referrer-Policy : same-origin\r\nX-Content-Type-Options : nosniff\r\nDate: Wed, 15 Oct 2025 15:13:12 GMT\r\n\r\n'
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/urllib3/connection.py", line 568, in getresponse
    assert_header_parsing(httplib_response.msg)
  File "/usr/local/lib/python3.12/dist-packages/urllib3/util/response.py", line 88, in assert_header_parsing
    raise HeaderParsingError(defects=defects, unparsed_data=unparsed_data)
urllib3.exceptions.HeaderParsingError: [MissingHeaderBodySeparato

  HEAD progress: 1000000/1000000
  Build progress: 1/1000000
  Build progress: 100001/1000000
  Build progress: 200001/1000000
  Build progress: 300001/1000000
  Build progress: 400001/1000000
  Build progress: 500001/1000000
  Build progress: 600001/1000000
  Build progress: 700001/1000000
  Build progress: 800001/1000000
